# 02 — LSH & Similarity Search in Spark (P1)
**Exam mapping:** 1a CRITIQUE an LLM's LSH design [15]; 1b implement `find_similar(text)` over a corpus by cosine [10].

The critique skill lives in notebook 01's style; this notebook gives you (a) the LSH parameter math to judge an answer against, and (b) a runnable, exam-ready `find_similar` in Spark.

## 1a — LSH design: the numbers to critique against
**Pick the family by similarity measure:**
| Similarity | Family | Hash | Collision prob |
|---|---|---|---|
| Jaccard | MinHash | min over random permutation | Pr = J |
| Cosine/angular | Random projection / SimHash | sign(r·v) | Pr = 1 − θ/π |
| Euclidean L2 | p-stable | ⌊(a·v+b)/w⌋ | decreasing in ‖u−v‖ |

**Banding / S-curve (amplification):** signature length n = b bands × r rows. `P(candidate | s) = 1 − (1 − s^r)^b`, threshold **t ≈ (1/b)^(1/r)**. ↑b → lower threshold (more recall, more false positives); ↑r → higher threshold (more precision).

**MinHash facts:** `Pr[minhash match] = J`; estimate Ĵ = matches/n; **Var = J(1−J)/n**; typical n = 100–200 → ~5% error.

**How to critique the past-exam LLM answer (steep S-curve, 60%→80% / 40%→5%):** right — random-projection is the correct cosine family; text must be vectorized (TF-IDF) first; the two base collision probs (0.70, 0.63) are close so strong amplification is needed. Wrong/limited — labeling random-hyperplane banding as "SimHash" is imprecise (SimHash = single compact fingerprint for Hamming near-dupes); the derived params (r=31, b=81,700, 2.5M-bit signature) are absurdly impractical and a good answer says so and relaxes the targets; it never *fixes* one text representation (TF-IDF vs embeddings change what cosine means).

In [ ]:
# VERIFIER for 1a: given target (b,r), print the S-curve and threshold so you can judge any claim.
import numpy as np
def scurve(b, r, s):  return 1 - (1 - s**r)**b
def threshold(b, r):  return (1.0/b)**(1.0/r)
print(f"{'(b,r)':>10} {'t≈(1/b)^(1/r)':>14}  P(cand) at s=0.3/0.5/0.7/0.9")
for b, r in [(20,5),(10,10),(50,2),(5,20)]:
    ps = [scurve(b,r,s) for s in (0.3,0.5,0.7,0.9)]
    print(f"{str((b,r)):>10} {threshold(b,r):>14.3f}  " + " ".join(f"{p:.3f}" for p in ps))
print("\nRule: to catch Jaccard>=0.7 pick (b,r) whose knee t sits near 0.7 (e.g. b=10,r=10 -> t=0.79).")

## 1b — `find_similar(text)` in Spark (TF-IDF + cosine)
Design choice to STATE: after cleaning, exam corpora are often small (the past exam had ~106 books) — an **exact cosine search over L2-normalized TF-IDF vectors is more reliable than building an LSH index**, and still honors the 1a cosine objective. For a genuinely large corpus you'd switch to MinHash/`BucketedRandomProjectionLSH` (built into `pyspark.ml.feature`), and pick (b,r)/bucketLength from the S-curve above.

Pipeline (`pyspark.ml`, all built-in): RegexTokenizer → StopWordsRemover → HashingTF → IDF → Normalizer(p=2). L2-normalized ⇒ cosine = dot product.

In [ ]:
import os
from pyspark.sql import SparkSession, Row
from pyspark.sql import functions as F
from pyspark.sql.types import DoubleType
from pyspark.ml import Pipeline
from pyspark.ml.feature import RegexTokenizer, StopWordsRemover, HashingTF, IDF, Normalizer
from pyspark.ml.linalg import SparseVector, DenseVector

spark = (SparkSession.builder.appName("sds-find-similar")
         .master("local[*]").config("spark.ui.enabled", "false").getOrCreate())
spark.sparkContext.setLogLevel("ERROR")

CORPUS_DIR = "data/corpus"      # <-- exam: the Gutenberg path; one canonical .txt per book id
# ---- load one canonical text per book id (dir name = id) ----
rows = []
for bid in sorted(os.listdir(CORPUS_DIR)):
    d = os.path.join(CORPUS_DIR, bid)
    if not os.path.isdir(d): continue
    cand = [f for f in os.listdir(d) if f.endswith(".txt") and "/old/" not in f]
    if not cand: continue
    path = os.path.join(d, sorted(cand, key=len)[0])
    with open(path, encoding="utf-8", errors="ignore") as fh:
        rows.append(Row(book_id=bid, text=fh.read()))
books = spark.createDataFrame(rows)
print("books:", books.count())

pipe = Pipeline(stages=[
    RegexTokenizer(inputCol="text", outputCol="tok", pattern="[^A-Za-z]+", minTokenLength=2, toLowercase=True),
    StopWordsRemover(inputCol="tok", outputCol="clean"),
    HashingTF(inputCol="clean", outputCol="tf", numFeatures=1 << 16),
    IDF(inputCol="tf", outputCol="tfidf"),
    Normalizer(inputCol="tfidf", outputCol="features", p=2.0),
])
model = pipe.fit(books)
book_vecs = model.transform(books).select("book_id", "features").cache()
book_vecs.count()

def _sparse_dot(v, q_idx, q_val):
    # manual dot over two sparse vectors — avoids SparseVector.dot (which calls
    # np.in1d, removed in numpy 2.x). Robust across numpy versions.
    if isinstance(v, DenseVector):
        return float(sum(v[i] * val for i, val in zip(q_idx, q_val)))
    d = dict(zip(v.indices.tolist(), v.values.tolist()))
    return float(sum(d.get(i, 0.0) * val for i, val in zip(q_idx, q_val)))

def find_similar(text):
    qv = model.transform(spark.createDataFrame([Row(text=text)])).select("features").first()["features"]
    q_idx = qv.indices.tolist(); q_val = qv.values.tolist()
    bi = spark.sparkContext.broadcast(q_idx); bvv = spark.sparkContext.broadcast(q_val)
    cos = F.udf(lambda v: _sparse_dot(v, bi.value, bvv.value), DoubleType())
    top = (book_vecs.withColumn("sim", cos(F.col("features")))
           .orderBy(F.desc("sim"), F.asc("book_id")).first())
    return top["book_id"]

print("query -> book:", find_similar("ocean voyage whale ship captain sailor storm"))

### What to report (1b)
- Justify exact-cosine-for-small vs LSH-for-large (tie back to 1a). State the pipeline + `numFeatures`. Corpus cleaning rules (drop `/old/`, pick canonical file per id from the parent-dir id). Return the top book id; optionally the similarity value.
- Gotcha: L2-normalize so cosine = dot product; HashingTF collisions shrink with larger `numFeatures`.
- **Library gotcha (seen in build):** Spark's `SparseVector.dot` calls `np.in1d`, removed in numpy 2.x — under a new numpy it throws `AttributeError`. The cell uses a manual sparse dot instead. If `.dot()` errors on Jojie, this is why; the manual version is the fallback.

In [ ]:
book_vecs.unpersist(); spark.stop(); print("done")